In [15]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import string
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

In [16]:
# Download necessary NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
    nltk.download('stopwords')
    nltk.download('wordnet')
    nltk.download('punkt_tab')


In [18]:
print("="*60)
print("SPAM DATASET CLEANING AND BALANCING")
print("="*60)

# Load the dataset
df = pd.read_csv('../data/spam_detection_dataset.csv')
print(f"\n📂 Dataset loaded successfully!")
print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")

SPAM DATASET CLEANING AND BALANCING

📂 Dataset loaded successfully!
   Shape: (6000, 2)
   Columns: ['text', 'label']


In [19]:

# Display class distribution
print("\n📊 Original Class Distribution:")
print(df['label'].value_counts())
print(f"\n   Spam: {len(df[df['label']=='spam'])} ({len(df[df['label']=='spam'])/len(df)*100:.2f}%)")
print(f"   Ham: {len(df[df['label']=='ham'])} ({len(df[df['label']=='ham'])/len(df)*100:.2f}%)")



📊 Original Class Distribution:
label
ham     3600
spam    2400
Name: count, dtype: int64

   Spam: 2400 (40.00%)
   Ham: 3600 (60.00%)


In [ ]:

if len(df[df['label']=='ham']) < 100:
    print("\n⚠️  Warning: Severe class imbalance detected!")
    print("   We need to balance the dataset for proper model training.")
    
    print("\n🔄 Balancing dataset...")
    
    df_spam = df[df['label'] == 'spam']
    df_ham = df[df['label'] == 'ham']
    
    if 'category' in df_ham.columns:
        ham_categories = df_ham['category'].unique()
        print(f"   Found {len(ham_categories)} ham categories: {list(ham_categories)}")
        
        n_ham_original = len(df_ham)
        n_to_generate = n_samples_needed - n_ham_original
        
        print(f"\n   Need to generate {n_to_generate} additional ham samples...")
        
        def augment_text(text, variations=1):
            """Create simple variations of text for augmentation"""
            augmented = [text]
            for _ in range(variations - 1):
                new_text = text
                
                replacements = {
                    'hello': ['hi', 'hey', 'greetings'],
                    'thanks': ['thank you', 'appreciate it', 'many thanks'],
                    'please': ['kindly', 'if you could'],
                    'meeting': ['appointment', 'session', 'gathering'],
                    'tomorrow': ['next day', 'day after'],
                }
                
                for word, options in replacements.items():
                    if word in new_text.lower():
                        new_text = new_text.replace(word, np.random.choice(options))
                
                if new_text != text and new_text not in augmented:
                    augmented.append(new_text)
            
            return augmented[1:] 
        
        augmented_hames = []
        
        variations_per_sample = max(1, n_to_generate // n_ham_original + 1)
        
        for idx, row in df_ham.iterrows():
            variations = augment_text(row['text'], variations_per_sample)
            for var_text in variations[:variations_per_sample]:  # Limit to needed count
                new_row = row.copy()
                new_row['text'] = var_text
                new_row['augmented'] = True
                augmented_hames.append(new_row)
                
                if len(augmented_hames) >= n_to_generate:
                    break
            if len(augmented_hames) >= n_to_generate:
                break
        
        # Create augmented dataframe
        df_ham_augmented = pd.DataFrame(augmented_hames[:n_to_generate])
        df_ham_augmented['augmented'] = True
        
        # Combine original ham with augmented
        df_ham_original = df_ham.copy()
        df_ham_original['augmented'] = False
        df_ham_combined = pd.concat([df_ham_original, df_ham_augmented], ignore_index=True)
        
        # Combine with spam
        df_spam['augmented'] = False
        df_balanced = pd.concat([df_spam, df_ham_combined], ignore_index=True)
        
        print(f"\n✅ Balanced dataset created!")
        print(f"   New shape: {df_balanced.shape}")
        print(f"   Spam: {len(df_balanced[df_balanced['label']=='spam'])}")
        print(f"   Ham: {len(df_balanced[df_balanced['label']=='ham'])}")
        print(f"   Ham samples from augmentation: {len(df_ham_augmented)}")
        
        df = df_balanced
    
    else:
        # Option 2: Simple upsampling with noise
        print("\n   Using SMOTE-like upsampling for ham class...")
        
        # Upsample ham class with slight noise
        df_ham_upsampled = resample(df_ham, 
                                     replace=True, 
                                     n_samples=len(df_spam), 
                                     random_state=42)
        
        # Combine with spam
        df_balanced = pd.concat([df_spam, df_ham_upsampled])
        
        # Add small noise to upsampled texts to avoid exact duplicates
        def add_noise(text, noise_level=0.05):
            """Add small noise to text"""
            if np.random.random() < noise_level:
                words = text.split()
                if len(words) > 3:
                    # Randomly swap two words
                    idx1, idx2 = np.random.choice(len(words), 2, replace=False)
                    words[idx1], words[idx2] = words[idx2], words[idx1]
                    return ' '.join(words)
            return text
        
        # Apply noise to upsampled ham
        upsampled_indices = df_balanced[df_balanced['label']=='ham'].index
        for idx in upsampled_indices:
            df_balanced.at[idx, 'text'] = add_noise(df_balanced.at[idx, 'text'])
        
        df = df_balanced
        print(f"\n✅ Balanced dataset created via upsampling!")
        print(f"   New shape: {df.shape}")
        print(f"   Spam: {len(df[df['label']=='spam'])}")
        print(f"   Ham: {len(df[df['label']=='ham'])}")

# Shuffle the balanced dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# ============================================
# TEXT CLEANING FUNCTIONS
# ============================================

def clean_text(text):
    """Basic text cleaning"""
    if pd.isna(text):
        return ""
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', ' URL ', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' EMAIL ', text)
    
    # Remove phone numbers
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', ' PHONE ', text)
    
    # Remove currency symbols
    text = re.sub(r'[$£€]\s?\d+(?:\.\d+)?', ' MONEY ', text)
    
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

def clean_text_advanced(text, remove_stopwords=True, lemmatize=True):
    """Advanced text cleaning with NLP options"""
    if pd.isna(text):
        return ""
    
    # Basic cleaning first
    text = clean_text(text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        custom_stops = {'u', 'ur', 'im', 'dont', 'cant', 'wont', 'get', 'via'}
        stop_words.update(custom_stops)
        tokens = [t for t in tokens if t not in stop_words]
    
    # Lemmatization
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    # Remove tokens that are just numbers or single characters
    tokens = [t for t in tokens if len(t) > 1 and not t.isdigit()]
    
    return ' '.join(tokens)

def extract_features_from_text(text):
    """Extract additional features from text"""
    text_lower = text.lower() if isinstance(text, str) else ""
    
    features = {
        'char_count': len(text) if isinstance(text, str) else 0,
        'word_count': len(text.split()) if isinstance(text, str) else 0,
        'digit_count': sum(c.isdigit() for c in text_lower) if isinstance(text, str) else 0,
        'uppercase_count': sum(c.isupper() for c in text) if isinstance(text, str) else 0,
        'exclamation_count': text.count('!') if isinstance(text, str) else 0,
        'question_count': text.count('?') if isinstance(text, str) else 0,
        'has_url': 1 if any(x in text_lower for x in ['http', 'www', '.com']) else 0,
        'has_money': 1 if any(x in text_lower for x in ['$', '£', '€', 'dollar']) else 0,
    }
    
    return features

# ============================================
# APPLY CLEANING
# ============================================

print("\n🧹 Applying text cleaning...")

# Apply cleaning
df['text_cleaned_basic'] = df['text'].apply(clean_text)
df['text_cleaned_advanced'] = df['text'].apply(lambda x: clean_text_advanced(x, remove_stopwords=True, lemmatize=True))

# Extract additional features
print("📊 Extracting additional text features...")
text_features = df['text'].apply(extract_features_from_text)
text_features_df = pd.DataFrame(text_features.tolist())
df = pd.concat([df, text_features_df], axis=1)

# ============================================
# EXPLORATORY DATA ANALYSIS
# ============================================

print("\n📈 Performing Exploratory Data Analysis...")

# 1. Class distribution after balancing
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
df['label'].value_counts().plot(kind='bar', color=['red', 'green'])
plt.title('Class Distribution After Balancing')
plt.xlabel('Label')
plt.ylabel('Count')
plt.xticks(rotation=45)

# 2. Text length distribution
plt.subplot(1, 3, 2)
df[df['label']=='spam']['word_count'].hist(alpha=0.7, label='Spam', bins=30, color='red')
df[df['label']=='ham']['word_count'].hist(alpha=0.7, label='Ham', bins=30, color='green')
plt.xlabel('Word Count')
plt.ylabel('Frequency')
plt.title('Word Count Distribution')
plt.legend()

# 3. Character count distribution
plt.subplot(1, 3, 3)
df[df['label']=='spam']['char_count'].hist(alpha=0.7, label='Spam', bins=30, color='red')
df[df['label']=='ham']['char_count'].hist(alpha=0.7, label='Ham', bins=30, color='green')
plt.xlabel('Character Count')
plt.ylabel('Frequency')
plt.title('Character Count Distribution')
plt.legend()

plt.tight_layout()
plt.savefig('spam_balanced_distribution.png')
plt.show()

# 4. Feature comparison
print("\n📊 Feature Statistics by Label (After Balancing):")
feature_cols = ['char_count', 'word_count', 'digit_count', 'uppercase_count', 
                'exclamation_count', 'question_count', 'has_url', 'has_money']

for col in feature_cols:
    if col in df.columns:
        spam_mean = df[df['label']=='spam'][col].mean()
        ham_mean = df[df['label']=='ham'][col].mean()
        print(f"   {col}: Spam={spam_mean:.2f}, Ham={ham_mean:.2f}")

# 5. Word Clouds
plt.figure(figsize=(15, 6))

# Spam word cloud
plt.subplot(1, 2, 1)
spam_text = ' '.join(df[df['label']=='spam']['text_cleaned_advanced'].dropna())
if spam_text.strip():
    wordcloud_spam = WordCloud(width=400, height=300, background_color='black', 
                               colormap='Reds', max_words=50).generate(spam_text)
    plt.imshow(wordcloud_spam, interpolation='bilinear')
    plt.title('Spam Messages - Most Common Words')
    plt.axis('off')

# Ham word cloud
plt.subplot(1, 2, 2)
ham_text = ' '.join(df[df['label']=='ham']['text_cleaned_advanced'].dropna())
if ham_text.strip():
    wordcloud_ham = WordCloud(width=400, height=300, background_color='white', 
                              colormap='Greens', max_words=50).generate(ham_text)
    plt.imshow(wordcloud_ham, interpolation='bilinear')
    plt.title('Ham Messages - Most Common Words')
    plt.axis('off')

plt.tight_layout()
plt.savefig('spam_wordclouds_balanced.png')
plt.show()

# ============================================
# SAVE PROCESSED DATASET
# ============================================

print("\n💾 Saving processed dataset...")

# Prepare final dataset
final_columns = ['text', 'text_cleaned_basic', 'text_cleaned_advanced', 'label'] + feature_cols
if 'category' in df.columns:
    final_columns.insert(1, 'category')
if 'augmented' in df.columns:
    final_columns.append('augmented')

available_final_cols = [col for col in final_columns if col in df.columns]
df_final = df[available_final_cols]

# Save to CSV
df_final.to_csv('spam_dataset_processed_balanced.csv', index=False)
print(f"   Saved as 'spam_dataset_processed_balanced.csv'")
print(f"   Shape: {df_final.shape}")

# ============================================
# FINAL SUMMARY
# ============================================

print("\n" + "="*60)
print("PROCESSING COMPLETE - SUMMARY")
print("="*60)
print(f"\n✅ Final dataset shape: {df_final.shape}")
print(f"✅ Final class distribution:")
print(df_final['label'].value_counts())
print(f"\n✅ Features extracted: {len(feature_cols)} text-based features")
print(f"✅ Visualizations saved:")
print(f"   - spam_balanced_distribution.png")
print(f"   - spam_wordclouds_balanced.png")

print(f"\n📊 Final Dataset Columns:")
for col in df_final.columns:
    print(f"   - {col}")

print(f"\n🎯 Next Steps:")
print(f"   1. Split data into train/test sets (e.g., 80/20 split)")
print(f"   2. Try different ML models:")
print(f"      - Using extracted features: Random Forest, XGBoost")
print(f"      - Using text: TF-IDF + Naive Bayes, Logistic Regression")
print(f"      - Deep learning: LSTM, Transformers")
print(f"   3. Evaluate using precision, recall, F1-score (not just accuracy)")

# Display sample
print(f"\n📝 Sample of Processed Data (First 3 rows):")
print(df_final.head(3).to_string())